# 00 — Setup do pipeline RenovaBio × AFOLU

Notebook de validação inicial. **Não faz processamento metodológico.** Apenas:

1. Monta o Google Drive
2. Adiciona o pacote `pipeline/` ao `sys.path`
3. Inventaria os arquivos em `data/raw/`
4. Valida que cada base abre corretamente e tem o shape esperado
5. Reporta cobertura por UF e janela temporal

Se algum passo falhar, o erro vai indicar exatamente qual arquivo está faltando ou problemático.

**Rode esta seção uma vez** depois que tiver montado a estrutura `data/raw/` no Drive.

In [ ]:
# Célula 1 — Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Célula 2 — Adicionar o pacote pipeline ao sys.path
#
# Estrutura esperada no Drive:
#   /content/drive/MyDrive/Renovabio - EcoEco/
#       pipeline/        <- módulos .py
#       data/raw/...     <- bases brutas
#       notebooks/       <- este notebook está aqui

import sys
from pathlib import Path

BASE_DIR = Path('/content/drive/MyDrive/Renovabio - EcoEco')

# Adiciona o diretório-raiz ao sys.path para que `import pipeline` funcione
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

# Sanity check
assert (BASE_DIR / 'pipeline').exists(), (
    f'Pasta pipeline/ não encontrada em {BASE_DIR}. '
    'Faça o upload do esqueleto antes de rodar este notebook.'
)
print(f'✓ BASE_DIR existe: {BASE_DIR}')
print(f'✓ pipeline/ existe: {(BASE_DIR / "pipeline").exists()}')
print(f'✓ data/raw/ existe: {(BASE_DIR / "data" / "raw").exists()}')

In [ ]:
# Célula 3 — Importar módulos do pipeline
from pipeline import config
from pipeline.config import (
    PARAMS, RAW_INPUTS, BASE_DIR,
    DATA_RAW, DATA_INTERIM, DATA_PROCESSED, OUTPUTS_PRE,
    ensure_dir,
)
from pipeline.io import file_size_mb, read_csv_smart, read_excel_safe
from pipeline.normalize import detect_encoding, detect_sep

import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print(f'pipeline version: {config.__version__ if hasattr(config, "__version__") else "?"}')
print(f'UFs core ({len(PARAMS.UFS_CORE)}): {PARAMS.UFS_CORE}')
print(f'Janela principal: {PARAMS.YEAR_MIN_MAIN}–{PARAMS.YEAR_MAX_MAIN}')
print(f'Janela full: {PARAMS.YEAR_MIN_FULL}–{PARAMS.YEAR_MAX_FULL}')

In [ ]:
# Célula 4 — Criar diretórios interim/, processed/ e outputs_pre/ se não existirem
for p in (DATA_INTERIM, DATA_PROCESSED, OUTPUTS_PRE):
    ensure_dir(p)
    print(f'✓ {p}')

## Inventário de `data/raw/`

In [ ]:
# Célula 5 — Inventário dos arquivos raw esperados
rows = []
for label, path in RAW_INPUTS.items():
    rows.append({
        'label': label,
        'path': str(path).replace(str(BASE_DIR), '.'),
        'exists': path.exists(),
        'size_mb': file_size_mb(path),
    })
df_inv = pd.DataFrame(rows)
n_ok = df_inv['exists'].sum()
n_total = len(df_inv)
print(f'{n_ok}/{n_total} arquivos encontrados\n')
df_inv

In [ ]:
# Célula 6 — Aborta se houver arquivos faltando
missing = df_inv[~df_inv['exists']]
if len(missing):
    print('⚠️  Arquivos faltando:')
    for _, r in missing.iterrows():
        print(f'  - {r.label}: {r.path}')
    print('\nFaça o upload destes arquivos para os caminhos indicados.')
    print('A validação por base nas próximas células vai pular os faltantes.')
else:
    print('✅ Todos os arquivos esperados estão presentes.')

## Validação por base

Para cada base raw, abrimos com encoding/separador detectado e reportamos:
- shape (linhas × colunas)
- cobertura geográfica (UFs)
- cobertura temporal (anos)

Erros aqui significam que o arquivo está corrompido ou no formato errado.

In [ ]:
# Célula 7a — Universo core 6 UFs
from pipeline.config import IBGE_UNIVERSO_CORE

if IBGE_UNIVERSO_CORE.exists():
    uc = pd.read_csv(IBGE_UNIVERSO_CORE)
    print(f'✓ Universo core: {uc.shape}')
    print(f'  UFs ({uc["uf"].nunique()}): {uc["uf"].value_counts().to_dict()}')
    # Validação contra IBGE oficial
    ibge_ref = {'MG': 853, 'SP': 645, 'PR': 399, 'GO': 246, 'MT': 141, 'MS': 79}
    diffs = []
    for uf, n_ref in ibge_ref.items():
        n_obs = (uc['uf'] == uf).sum()
        if n_obs != n_ref:
            diffs.append(f'{uf}: obs={n_obs}, IBGE={n_ref}, diff={n_obs - n_ref:+d}')
    if diffs:
        print('\n  ⚠️  Divergências com IBGE:')
        for d in diffs: print(f'    {d}')
    else:
        print(f'  ✅ Total {len(uc)} bate com IBGE (2.363)')

In [ ]:
# Célula 7b — PAM tabela 1612 (formato SIDRA empilhado)
from pipeline.config import IBGE_PAM_1612

if IBGE_PAM_1612.exists():
    with open(IBGE_PAM_1612, 'r', encoding='utf-8', errors='replace') as f:
        lines = f.readlines()
    blocos = [(i, l.strip()[:120]) for i, l in enumerate(lines) if l.startswith('"Variável -')]
    print(f'✓ PAM tabela 1612: {len(lines):,} linhas, {len(blocos)} blocos de variável')
    for i, txt in blocos:
        print(f'  [linha {i:>5}] {txt[:100]}')
    # Janela temporal
    if len(lines) > 4:
        # linha 3 contém anos
        anos = [t.strip().strip('"') for t in lines[3].split(',') if t.strip().strip('"').isdigit()]
        print(f'  Janela: {anos[0]}–{anos[-1]} ({len(anos)} anos)')

In [ ]:
# Célula 7c — Certificados ANP (3 snapshots)
from pipeline.config import ANP_CERT_2022, ANP_CERT_2025, ANP_CERT_2026

for label, path in [('2022', ANP_CERT_2022), ('2025', ANP_CERT_2025), ('2026', ANP_CERT_2026)]:
    if not path.exists():
        print(f'  ⏭️  {label}: arquivo ausente, pulando')
        continue
    xl = pd.ExcelFile(path)
    sheets_relevantes = ['Válidos', 'Cancelados ou Suspensos', 'Anulados']
    print(f'\n✓ ANP {label}: {path.name}')
    for s in sheets_relevantes:
        if s in xl.sheet_names:
            df = pd.read_excel(path, sheet_name=s, header=1).dropna(how='all')
            print(f'    {s}: {df.shape[0]} linhas')

In [ ]:
# Célula 7d — NEEA consolidado (do TCC)
from pipeline.config import ANP_NEEA

if ANP_NEEA.exists():
    neea = pd.read_csv(ANP_NEEA)
    n_neea_cols = len([c for c in neea.columns if 'neea' in c.lower()])
    n_vol_cols  = len([c for c in neea.columns if c.lower().startswith('vol_')])
    print(f'✓ NEEA consolidado: {neea.shape}')
    print(f'  Colunas NEEA-like: {n_neea_cols}, vol-like: {n_vol_cols}')
    # Cobertura UF (parsed do cidade_uf)
    if 'cidade_uf' in neea.columns:
        import re as _re
        ufs = neea['cidade_uf'].astype(str).str.extract(r'/\s*([A-Z]{2})\s*$', expand=False)
        print(f'  UFs (parsed): {ufs.value_counts(dropna=False).to_dict()}')

In [ ]:
# Célula 7e — SEEG por UF (6 arquivos)
from pipeline.config import SEEG_FILES

for uf, path in SEEG_FILES.items():
    if not path.exists():
        print(f'  ⏭️  {uf}: arquivo ausente')
        continue
    enc = detect_encoding(path)
    sep = detect_sep(path, encoding=enc)
    # leitura amostra para diagnóstico (dtype=str para não tentar parsear nada)
    df = pd.read_csv(path, encoding=enc, sep=sep, dtype=str, low_memory=False, nrows=3000)
    year_cols = [c for c in df.columns if str(c).strip().isdigit() and 1900 <= int(c.strip()) <= 2100]
    if year_cols:
        print(f'✓ {uf}: {file_size_mb(path):>6.2f} MB | enc={enc} sep="{sep}" | '
              f'{len(df)}+ linhas | {len(year_cols)} cols-ano ({year_cols[0]}–{year_cols[-1]})')
    else:
        print(f'⚠️  {uf}: cols-ano não detectadas')

In [ ]:
# Célula 7f — MapBiomas (panel ready é input principal; raw fica como backup)
from pipeline.config import MAPBIOMAS_PANEL, MAPBIOMAS_RAW

if MAPBIOMAS_PANEL.exists():
    # IMPORTANTE: encoding=latin-1, sep=';', decimal_br para parser correto
    mb = pd.read_csv(MAPBIOMAS_PANEL, encoding='latin-1', sep=';', low_memory=False)
    print(f'✓ MapBiomas panel ready: {mb.shape}')
    if 'state_acronym' in mb.columns:
        print(f'  UFs: {mb["state_acronym"].value_counts().to_dict()}')
    if 'year' in mb.columns:
        anos = sorted(mb["year"].unique())
        print(f'  Janela: {anos[0]}–{anos[-1]} ({len(anos)} anos)')

if MAPBIOMAS_RAW.exists():
    print(f'✓ MapBiomas raw (backup): {file_size_mb(MAPBIOMAS_RAW):.2f} MB')

In [ ]:
# Célula 7g — SICAR painel mensal (~30 MB; carrega só a metadados primeiro)
from pipeline.config import SICAR_PAINEL

if SICAR_PAINEL.exists():
    print(f'✓ SICAR: {file_size_mb(SICAR_PAINEL):.2f} MB')
    # Lê só primeiros mil registros para validar shape sem carregar tudo
    sicar_sample = pd.read_excel(SICAR_PAINEL, nrows=1000)
    print(f'  Colunas ({len(sicar_sample.columns)}): {list(sicar_sample.columns)}')
    # Validação completa fica para o módulo sicar.py

In [ ]:
# Célula 7h — PSM baseline (covariáveis socioeconômicas)
from pipeline.config import PSM_BASELINE_RAW

if PSM_BASELINE_RAW.exists():
    psm = pd.read_csv(PSM_BASELINE_RAW, low_memory=False)
    print(f'✓ PSM baseline: {psm.shape}')
    n_critical = sum(1 for c in psm.columns if c.startswith('17_'))
    print(f'  Colunas grupo 17 (IDHM/IVS/Gini): {n_critical}')
    if '0_sg_uf' in psm.columns:
        in_core = psm['0_sg_uf'].isin(PARAMS.UFS_CORE).sum()
        print(f'  Linhas em 6 UFs Centro-Sul: {in_core}/{len(psm)}')

## Resumo

Se todos os ✓ apareceram acima, a infraestrutura básica está pronta. Próximos passos:

1. `01_crosswalk.ipynb` — construção do crosswalk universal a partir do `01_universo_core_6ufs.csv`.
2. `02_anp.ipynb` — adapta os 5 ajustes cirúrgicos do `renovabio_v2.py`.
3. `03_seeg.ipynb` — reconstrução dos outcomes AFOLU com Emissão+Remoção e `asinh()` em carbono_solo.

Cada notebook é independente. Se algo falhar no meio, basta corrigir o módulo e re-rodar — os interim/ anteriores ficam intactos.